# ULTRON owner setup — preflight only

**This notebook does not train.** It does not instantiate the model, create an optimizer, generate or download a dataset, perform optimizer steps, or dispatch remote jobs.

Purpose: confirm the repository is present, load the website-exported schema-v1 runtime configuration, and independently calculate the foundation model's parameter counts from configuration formulas. The notebook now verifies an existing dataset and constructs read-only split DataLoaders; checkpoint resume and training remain disabled.

The config file is untrusted input. Its authorization flags must remain false.

In [ ]:
from pathlib import Path
import json
import platform
import sys

repo_root = Path.cwd()
if not (repo_root / "ultron_core").is_dir():
    raise FileNotFoundError("Run this notebook from the ULTRON repository root (ultron_core/ must exist).")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print({"python": platform.python_version(), "platform": platform.platform(), "repo_root": str(repo_root)})

## Load the website-exported config

On the device/browser, export `ultron-runtime-config.json` from ULTRON Settings, then upload it to this notebook runtime using its file-upload UI. Enter the runtime-local path below. This is a manual file handoff; the website does not connect to this notebook.

In [ ]:
from ultron_core.runtime_config import RuntimeConfig, MAX_CONFIG_JSON_BYTES

config_path = Path(input("Runtime-local path to ultron-runtime-config.json: ").strip()).expanduser()
if not config_path.is_file():
    raise FileNotFoundError(f"Config file not found: {config_path}")
config_bytes = config_path.read_bytes()
if len(config_bytes) > MAX_CONFIG_JSON_BYTES:
    raise ValueError("Config exceeds the schema's 16 KiB limit.")
runtime_config = RuntimeConfig.from_json(config_bytes.decode("utf-8"))
assert runtime_config.training_authorized is False
assert runtime_config.remote_execution_enabled is False
print(json.dumps(runtime_config.to_dict(), indent=2, sort_keys=True))

## Capacity formula check — no model allocation

This constructs only the small immutable configuration object. The count functions use parameter-shape formulas; they do not instantiate `NeuroSparseModel` or allocate its weights.

In [ ]:
from ultron_core.config import NeuroSparseConfig, count_parameters, count_total_parameters, MIN_FOUNDATION_TRAINABLE_PARAMETERS

foundation_config = NeuroSparseConfig.foundation()
trainable_count = count_parameters(foundation_config)
total_count = count_total_parameters(foundation_config)
assert trainable_count >= MIN_FOUNDATION_TRAINABLE_PARAMETERS
assert total_count >= trainable_count
assert foundation_config.vocab_size == 240
print({
    "total_parameters_by_formula": total_count,
    "trainable_parameters_by_formula": trainable_count,
    "minimum_trainable_parameters": MIN_FOUNDATION_TRAINABLE_PARAMETERS,
    "sequence_length_limit": foundation_config.max_sequence_length,
    "runtime_budget_gib_requested": runtime_config.memory_budget_gib,
    "batch_size_requested": runtime_config.batch_size,
    "sequence_length_requested": runtime_config.sequence_length,
    "gradient_accumulation_requested": runtime_config.gradient_accumulation_steps,
})

## Verify an existing dataset and prepare split metadata

Upload an **already-existing** ULTRON JSONL dataset and its `.manifest.json` sidecar. This verifies the dataset and writes only deterministic split-ID metadata. It does not create or edit dataset rows, download data, or train.

In [ ]:
from ultron_core.dataset_split_manifest import prepare_dataset_split_manifest, validate_dataset_split_manifest

existing_dataset_path = Path(input("Path to existing verified ULTRON JSONL dataset: ").strip()).expanduser()
split_manifest_path = Path(input("Output path for split-ID metadata JSON: ").strip()).expanduser()
split_seed_text = input("Deterministic split seed [0]: ").strip()
split_seed = int(split_seed_text) if split_seed_text else 0
split_contract = prepare_dataset_split_manifest(existing_dataset_path, split_manifest_path, split_seed=split_seed, validation_fraction=0.1, test_fraction=0.1)
split_contract = validate_dataset_split_manifest(existing_dataset_path, split_manifest_path)
print({"dataset_sha256": split_contract.dataset_sha256, "split_manifest_sha256": split_contract.split_manifest_sha256, "dataset_examples": split_contract.dataset_example_count, "selected_unique_prompt_examples": split_contract.selected_example_count, "split_counts": split_contract.split_counts, "status": "DATASET_AND_SPLITS_VERIFIED / PREFLIGHT_ONLY"}

## Construct read-only split loaders (no training)

Build the train sampler/DataLoader and ordered validation/test loaders from the verified dataset. This cell does not iterate over batches or instantiate the model/optimizer. It records the initial cursor metadata shape for a future checkpoint callback.

In [ ]:
from ultron_core.verified_dataset import build_verified_split_loader

sampler_seed = split_seed % (2**63)
train_loader = build_verified_split_loader(existing_dataset_path, split_manifest_path, split_name="train", batch_size=runtime_config.batch_size, seed=sampler_seed, max_sequence_length=runtime_config.sequence_length)
validation_loader = build_verified_split_loader(existing_dataset_path, split_manifest_path, split_name="validation", batch_size=runtime_config.batch_size, seed=sampler_seed, max_sequence_length=runtime_config.sequence_length)
test_loader = build_verified_split_loader(existing_dataset_path, split_manifest_path, split_name="test", batch_size=runtime_config.batch_size, seed=sampler_seed, max_sequence_length=runtime_config.sequence_length)
train_sampler = train_loader.sampler
initial_data_state = {"epoch": 0, "next_batch_index": 0, "next_sample_index": 0, "dataset_manifest_sha256": train_loader.dataset_manifest_sha256, "sampler_state": train_sampler.state_dict()}
assert train_loader.num_workers == validation_loader.num_workers == test_loader.num_workers == 0
assert train_loader.drop_last is validation_loader.drop_last is test_loader.drop_last is False
assert train_sampler.dataset_manifest_sha256 == split_contract.dataset_sha256
print({"train_examples": len(train_loader.dataset), "validation_examples": len(validation_loader.dataset), "test_examples": len(test_loader.dataset), "train_sampler_state": train_sampler.state_dict(), "initial_checkpoint_data_state_shape": initial_data_state, "status": "SPLIT_LOADERS_CONSTRUCTED / NO_BATCHES_CONSUMED / NO_TRAINING"}

## Stop point

Expected result: `CONFIG_VALIDATED / DATASET_AND_SPLITS_VERIFIED / SPLIT_LOADERS_CONSTRUCTED / NO_BATCHES_CONSUMED / PREFLIGHT_ONLY`.

This notebook generates **no dataset examples** and downloads no data. It writes split-ID metadata only, verifies the existing corpus, and constructs read-only split datasets/DataLoaders. It does not iterate batches, allocate model weights, create an optimizer, restore a checkpoint, perform optimizer steps, train, or dispatch remote jobs. Do not mark the training gate complete.

Next engineering stage: pass the verified train loader/sampler into the existing bounded trainer and wire checkpoint callback `data_state`. Actual trainer execution and clean-runtime equivalence require separate explicit authorization for a tiny optimizer-step test.